In [ ]:
from dep_tools.searchers import PystacSearcher
from dep_tools.loaders import OdcLoader
from src.utils import SDBProcessor, S2_BANDS

from dep_tools.grids import PACIFIC_GRID_10
import joblib

In [ ]:
catalog = "https://earth-search.aws.element84.com/v1"
collection = "sentinel-2-l2a"

tile_id = (130, 12)
geobox = PACIFIC_GRID_10.tile_geobox(tile_id)
datetime = "2024-12-01/2024-12-05"

model = joblib.load("models/2025_03_12_randomforest_land_mask_30m.joblib")

searcher = PystacSearcher(
    catalog=catalog,
    collections=[collection],
    datetime=datetime,
    query={"eo:cloud_cover": {"lt": 100}},
)

loader = OdcLoader(
    bands=S2_BANDS,
    chunks={"x": 3201, "y": 3201},
    groupby="solar_day",
    fail_on_error=False,
)

processor = SDBProcessor(
    model=model,
    preprocessor_args={
        "mask_clouds": True,
    },
)

In [ ]:
items = searcher.search(geobox)

print(f"Found {len(items)} items")

In [ ]:
data = loader.load(items, geobox)

data

In [ ]:
results = processor.process(data)

results